In [1]:
# this line makes figures interactive in Jupyter notebooks
%matplotlib inline
from matplotlib import pyplot as plt

import numpy as np
import cantera as ct

from pint import UnitRegistry
ureg = UnitRegistry()
Q_ = ureg.Quantity

# for convenience:
def to_si(quant):
    '''Converts a Pint Quantity to magnitude at base SI units.
    '''
    return quant.to_base_units().magnitude

In [2]:
from thermo import get_thermo_derivatives, get_thermo_properties

In [3]:
o_f_ratio = 6.0
temperature_h2 = Q_(20.270, 'K')
temperature_o2 = Q_(90.170, 'K')
pressure_chamber = Q_(3000, 'psi')

h2o2_filename = "./h2o2_react.yaml"

h2 = ct.Solution(h2o2_filename, 'liquid_hydrogen')
h2.TP = to_si(temperature_h2), to_si(pressure_chamber)

o2 = ct.Solution(h2o2_filename, 'liquid_oxygen')
o2.TP = to_si(temperature_o2), to_si(pressure_chamber)

molar_ratio = o_f_ratio / (o2.mean_molecular_weight / h2.mean_molecular_weight)
moles_ox = molar_ratio / (1 + molar_ratio)
moles_f = 1 - moles_ox

gas2 = ct.Solution('nasa_h2o2.yaml', 'gas')

# create a mixture of the liquid phases with the gas-phase model,
# with the number of moles for fuel and oxidizer based on
# the O/F ratio
mix = ct.Mixture([(h2, moles_f), (o2, moles_ox), (gas2, 0)])

# Solve for the equilibrium state, at constant enthalpy and pressure
mix.equilibrate('HP')

gas2()
derivs = get_thermo_derivatives(gas2)

dlogV_dlogT_P, dlogV_dlogP_T, cp, gamma_s = get_thermo_properties(
    gas2, derivs[0], derivs[1], derivs[2]
    )

print(f'Cp = {cp: .2f} J/(K kg)')

print(f'(d log V/d log P)_T = {dlogV_dlogP_T: .4f}')
print(f'(d log V/d log T)_P = {dlogV_dlogT_P: .4f}')

print(f'gamma_s = {gamma_s: .4f}')

speed_sound = np.sqrt(ct.gas_constant * gas2.T * gamma_s / gas2.mean_molecular_weight)
print(f'Speed of sound = {speed_sound: .1f} m/s')


  gas:

       temperature   3597.5 K
          pressure   2.0684e+07 Pa
           density   9.4137 kg/m^3
  mean mol. weight   13.613 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -9.8628e+05       -1.3426e+07  J
   internal energy       -3.1835e+06       -4.3338e+07  J
           entropy             17175        2.3381e+05  J/K
    Gibbs function       -6.2775e+07       -8.5457e+08  J
 heat capacity c_p            3795.4             51668  J/K
 heat capacity c_v            3184.7             43354  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                 H         0.0018901          0.025526           -8.7917
               HO2        8.3393e-05        3.4395e-05           -40.623
                H2          0.036632           0.24736           -17.583

In [4]:
from rocket import calculate_c_star

In [5]:
area_ratio = 68.8

pressure_throat_cea = Q_(118.85, 'bar').to('Pa')
temperature_throat_cea = 3381.67
pressure_exit_cea = Q_(0.21521, 'bar').to('Pa')
temperature_exit_cea = 1233.84
c_star_cea = 2322.8
thrust_coeff_cea = 1.8823
specific_impulse_cea = 4372.3
specific_impulse_vac_cea = 4538.6

In [6]:
entropy_chamber = gas2.s
enthalpy_chamber = gas2.enthalpy_mass
mole_fractions_chamber = gas2.X
gamma_chamber = gamma_s

c_star = calculate_c_star(gamma_chamber, gas2.T, gas2.mean_molecular_weight)
print(f'c-star: {c_star: .1f} m/s')
print('Error in c-star: '
      f'{100*np.abs(c_star - c_star_cea)/c_star_cea: .3e} %'
      )

c-star:  2323.0 m/s
Error in c-star:  7.130e-03 %


In [7]:
gas_throat = ct.Solution('nasa_h2o2.yaml', 'gas')

pressure_throat = pressure_chamber / np.power(
    (gamma_chamber + 1) / 2., gamma_chamber / (gamma_chamber - 1)
    )

# based on CEA defaults
max_iter_throat = 5
tolerance_throat = 0.4e-4

print('Throat iterations:')
mach = 1.0
num_iter = 0
residual = 1
while residual > tolerance_throat:
    num_iter += 1
    if num_iter == max_iter_throat:
        break
        print(f'Error: more than {max_iter_throat} iterations required for throat calculation')
    pressure_throat = pressure_throat * (1 + gamma_s * mach**2) / (1 + gamma_s)
    
    gas_throat.SPX = entropy_chamber, to_si(pressure_throat), mole_fractions_chamber
    gas_throat.equilibrate('SP')

    derivs = get_thermo_derivatives(gas_throat)
    dlogV_dlogT_P, dlogV_dlogP_T, cp, gamma_s = get_thermo_properties(
        gas_throat, derivs[0], derivs[1], derivs[2]
        )
    
    velocity = np.sqrt(2 * (enthalpy_chamber - gas_throat.enthalpy_mass))
    speed_sound = np.sqrt(
        ct.gas_constant * gas_throat.T * gamma_s / gas_throat.mean_molecular_weight
        )
    mach = velocity / speed_sound

    residual = np.abs(1.0 - 1/mach**2)
    print(f'{num_iter}  {residual: .3e}')

temperature_throat = gas_throat.T
pressure_throat = Q_(gas_throat.P, 'Pa')
gamma_s_throat = gamma_s

print('Error in throat temperature: '
      f'{100*np.abs(temperature_throat - temperature_throat_cea)/temperature_throat_cea: .3e} %'
      )
print('Error in throat pressure: '
      f'{100*np.abs(pressure_throat - pressure_throat_cea)/pressure_throat_cea: .3e~P} %'
      )

Throat iterations:
1   9.420e-04
2   1.590e-06
Error in throat temperature:  2.640e-02 %
Error in throat pressure: 5.430e-03 %


In [8]:
# this is constant
A_mdot_thr = gas_throat.T / (gas_throat.P * velocity * gas_throat.mean_molecular_weight)

gas_exit = ct.Solution('nasa_h2o2.yaml', 'gas')
gas_exit.SPX = gas_throat.s, gas_throat.P, gas_throat.X

# initial estimate for pressure ratio
pinf_pe = np.exp(gamma_s_throat + 1.4 * np.log(area_ratio))
p_exit = to_si(pressure_chamber) / pinf_pe

gas_exit.SP = entropy_chamber, p_exit
gas_exit.equilibrate('SP')

Ae_At = gas_exit.T / (gas_exit.P * velocity * gas_exit.mean_molecular_weight) / A_mdot_thr

print('Iter  T_exit   Ae/At    P_exit     P_inf/P')
num_iter = 0
print(f'{num_iter}  {gas_exit.T:.3f} K   {Ae_At: .2f}  {gas_exit.P/1e5:.3f} bar  {pinf_pe:.3f}')

max_iter_exit = 10
tolerance_exit = 4e-5

residual = 1
while np.abs(residual) > tolerance_exit:
    num_iter += 1
    
    if num_iter == max_iter_throat:
        break
        print(f'Error: more than {max_iter_exit} iterations required for exit calculation')

    derivs = get_thermo_derivatives(gas_exit)
    dlogV_dlogT_P, dlogV_dlogP_T, cp, gamma_s = get_thermo_properties(
        gas_exit, derivs[0], derivs[1], derivs[2]
        )
    velocity = np.sqrt(2 * (enthalpy_chamber - gas_exit.enthalpy_mass))
    speed_sound = np.sqrt(ct.gas_constant * gas_exit.T * gamma_s / gas_exit.mean_molecular_weight)

    Ae_At = gas_exit.T / (gas_exit.P * velocity * gas_exit.mean_molecular_weight) / A_mdot_thr

    dlogp_dlogA = gamma_s * velocity**2 / (velocity**2 - speed_sound**2)
    residual = dlogp_dlogA * (np.log(area_ratio) - np.log(Ae_At))
    log_pinf_pe = np.log(pinf_pe) + residual

    pinf_pe = np.exp(log_pinf_pe)
    p_exit = to_si(pressure_chamber) / pinf_pe

    gas_exit.SP = entropy_chamber, p_exit
    gas_exit.equilibrate('SP')
    
    print(f'{num_iter}  {gas_exit.T:.3f} K  {Ae_At: .2f}  {gas_exit.P/1e5:.3f} bar  {pinf_pe:.3f}')

Iter  T_exit   Ae/At    P_exit     P_inf/P
0  1183.787 K    230.93  0.175 bar  1178.875
1  1234.196 K   80.36  0.215 bar  960.727
2  1234.177 K   68.80  0.215 bar  960.800
3  1234.177 K   68.80  0.215 bar  960.800


In [9]:
derivs = get_thermo_derivatives(gas_exit)
dlogV_dlogT_P, dlogV_dlogP_T, cp, gamma_s = get_thermo_properties(
    gas_exit, derivs[0], derivs[1], derivs[2]
    )
velocity = np.sqrt(2 * (enthalpy_chamber - gas_exit.enthalpy_mass))

thrust_coeff = velocity / c_star
print(f'Thrust coefficient: {thrust_coeff: .4f}')

g0 = 9.80665
Isp = velocity
Ivac = Isp + gas_exit.T * ct.gas_constant / (velocity * gas_exit.mean_molecular_weight)
print(f'I_sp = {Isp: .1f} m/s')
print(f'I_vac = {Ivac: .1f} m/s')

print()
print('Error in Isp: '
      f'{100*np.abs(Isp - specific_impulse_cea)/specific_impulse_cea: .3e} %'
      )
print('Error in Ivac: '
      f'{100*np.abs(Ivac - specific_impulse_vac_cea)/specific_impulse_vac_cea: .3e} %'
      )

Thrust coefficient:  1.8822
I_sp =  4372.2 m/s
I_vac =  4538.5 m/s

Error in Isp:  2.903e-03 %
Error in Ivac:  2.513e-03 %


In [10]:
print('Actual specific impulse:')
print(f'I_sp = {Isp / g0: .1f} s')
print(f'I_vac = {Ivac / g0: .1f} s')

Actual specific impulse:
I_sp =  445.8 s
I_vac =  462.8 s


In [11]:
gas1 = ct.Solution('gri30.yaml')
gas1()


  gri30:

       temperature   300 K
          pressure   1.0133e+05 Pa
           density   0.081894 kg/m^3
  mean mol. weight   2.016 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy             26469             53361  J
   internal energy       -1.2108e+06        -2.441e+06  J
           entropy             64910        1.3086e+05  J/K
    Gibbs function       -1.9447e+07       -3.9204e+07  J
 heat capacity c_p             14311             28851  J/K
 heat capacity c_v             10187             20536  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                H2                 1                 1           -15.717
     [  +52 minor]                 0                 0  



In [12]:
gas1.TPX = 300.0, ct.one_atm, 'CH4:095, O2:2, N2:7.52'
gas1()


  gri30:

       temperature   300 K
          pressure   1.0132e+05 Pa
           density   0.69909 kg/m^3
  mean mol. weight   17.21 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -3.9362e+06        -6.774e+07  J
   internal energy       -4.0811e+06       -7.0234e+07  J
           entropy             11054        1.9024e+05  J/K
    Gibbs function       -7.2525e+06       -1.2481e+08  J
 heat capacity c_p            2042.9             35158  J/K
 heat capacity c_v            1559.8             26843  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                O2          0.035578          0.019135            -28.63
               CH4            0.8473           0.90892           -52.418
                N2           0.11712          0.071948           -25.665


In [13]:
gas1.equilibrate('HP')

In [14]:
gas1()


  gri30:

       temperature   490.37 K
          pressure   1.0133e+05 Pa
           density   0.42696 kg/m^3
  mean mol. weight   17.18 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -3.9362e+06       -6.7623e+07  J
   internal energy       -4.1735e+06         -7.17e+07  J
           entropy             12215        2.0985e+05  J/K
    Gibbs function        -9.926e+06       -1.7053e+08  J
 heat capacity c_p            2588.2             44464  J/K
 heat capacity c_v            2104.2             36150  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                H2        0.00040455         0.0034475           -21.757
               H2O          0.018216          0.017372           -86.507
               CH4           0.83756           0.89692           -41.3

In [32]:
o_f_ratio = 6.0
temperature_h2 = Q_(20.270, 'K')
temperature_o2 = Q_(90.170, 'K')
pressure_chamber = Q_(3000, 'psi')

h2o2_filename = "./h2o2_react.yaml"

h2 = ct.Solution(h2o2_filename, 'liquid_hydrogen')
h2.TP = to_si(temperature_h2), to_si(pressure_chamber)

o2 = ct.Solution(h2o2_filename, 'liquid_oxygen')
o2.TP = to_si(temperature_o2), to_si(pressure_chamber)

molar_ratio = o_f_ratio / (o2.mean_molecular_weight / h2.mean_molecular_weight)
moles_ox = molar_ratio / (1 + molar_ratio)
moles_f = 1 - moles_ox

full_species = {S.name: S for S in ct.Species.list_from_file('nasa_gas.yaml')}
species = [full_species[S] for S in ('H', 'HO2', 'H2', 'H2O', 'H2O2', 'O', 'OH', 'O2', 'O3')]
# condensed_species = {S.name: S for S in ct.Species.list_from_file('nasa_condensed.yaml')}
# species += [condensed_species[S] for S in ('H2O(s)', 'H2O(L)')]
gas2 = ct.Solution(thermo='IdealGas', species=full_species.values())

# create a mixture of the liquid phases with the gas-phase model,
# with the number of moles for fuel and oxidizer based on
# the O/F ratio
mix = ct.Mixture([(h2, moles_f), (o2, moles_ox), (gas2, 0)])

# Solve for the equilibrium state, at constant enthalpy and pressure
mix.equilibrate('HP')

gas2()
derivs = get_thermo_derivatives(gas2)

dlogV_dlogT_P, dlogV_dlogP_T, cp, gamma_s = get_thermo_properties(
    gas2, derivs[0], derivs[1], derivs[2]
    )

print(f'Cp = {cp: .2f} J/(K kg)')

print(f'(d log V/d log P)_T = {dlogV_dlogP_T: .4f}')
print(f'(d log V/d log T)_P = {dlogV_dlogT_P: .4f}')

print(f'gamma_s = {gamma_s: .4f}')

speed_sound = np.sqrt(ct.gas_constant * gas2.T * gamma_s / gas2.mean_molecular_weight)
print(f'Speed of sound = {speed_sound: .1f} m/s')


       temperature   20.27 K
          pressure   2.0684e+07 Pa
           density   3192.8 kg/m^3
  mean mol. weight   26.015 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -7.6509e+06       -1.9903e+08  J
   internal energy       -7.6574e+06        -1.992e+08  J
           entropy            3002.5             78110  J/K
    Gibbs function       -7.7118e+06       -2.0062e+08  J
 heat capacity c_p            1347.4             35053  J/K
 heat capacity c_v            1027.8             26738  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
          Electron        1.0226e-18        4.8493e-14           -55.409
               H2O           0.34625               0.5           -1497.5
              H2O2           0.65375               0.5           -883.29
     [ +